# Fase 6.2 — Potencia hacia adelante, protocolo de monitoreo y despliegue

**Tesis v2 · Eider Johan Díaz Berrío · MCDA EAFIT**

Este notebook cierra la Fase 6. Contesta tres preguntas prácticas:

1. ¿Cuántos eventos maduros harían falta para **detectar** que adaptar mejora respecto al modelo estático?
2. ¿Cómo goberno el modelo en producción? (protocolo de monitoreo con disparadores)
3. ¿Qué hago con el despliegue AWS que ya funciona? (documentar, no redesplegar)

No rehago el servicio en la nube: la decisión 5 del Project deja AWS intacto hasta ver los resultados v2.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..", "src").resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openpyxl import load_workbook

from tesis_v2 import adaptativo as ad, config, contexto, figuras, io_datos, estadistica as est

config.asegurar_carpetas()
figuras.estilo()
np.random.seed(config.SEMILLA)

D, ctx, ds = contexto.cargar()
res = pd.read_csv(config.RESULTADOS / "fase6" / "metricas_estrategias.csv")
comp = pd.read_csv(config.RESULTADOS / "fase6" / "comparaciones_pareadas.csv")
print(res[["estrategia", "n", "eventos", "log_loss", "auc"]].round(4).to_string(index=False))

      estrategia   n  eventos  log_loss    auc
bayes_secuencial 433       10    0.1556 0.5028
        estatico 433       10    0.1336 0.6338
       expansivo 433       10    0.1257 0.5300
       ponderado 433       10    0.1256 0.5225
       tasa_base 433       10    0.1203 0.4661


## 1. Potencia hacia adelante

Con la tasa de eventos observada por año de originación proyecté cuántos incumplimientos adicionales se necesitan para detectar mejoras de AUC de 0,05 a 0,15 (80 % de potencia, α = 0,05, correlación entre AUC r = 0,5, Hanley–McNeil).

In [2]:
ev = int(D["y"].sum())
n0 = len(D) - ev
prev = D["y"].mean()
ev_anio = ad.tasa_eventos_anual(D)
print(f"Eventos actuales: {ev}; no eventos: {n0}; prevalencia: {100*prev:.2f} %")
print(f"Tasa media de eventos por año de originación: {ev_anio:.2f}")

pot = ad.potencia_adelante(ev, n0, prev, auc_ref=0.65, eventos_por_anio=ev_anio)
io_datos.guardar_resultado(pot, "fase6", "potencia_adelante")
pot.round(3)

Eventos actuales: 23; no eventos: 710; prevalencia: 3.14 %
Tasa media de eventos por año de originación: 5.75


,delta_auc,eventos_necesarios,eventos_adicionales,anios_aprox,se_delta_en_necesarios,potencia_actual_%
0,0.05,288,265,46.087,0.018,12.143
1,0.08,113,90,15.652,0.028,24.427
2,0.10,72,49,8.522,0.036,35.354
3,0.15,32,9,1.565,0.054,66.141


In [3]:
# I03 — eventos adicionales vs Δ AUC detectable
fig, ax = plt.subplots(figsize=(7.2, 4.0))
ax.plot(pot["delta_auc"], pot["eventos_necesarios"], marker="o", color=figuras.AZUL, lw=1.5)
ax.axhline(ev, color=figuras.ROJO, ls="--", lw=1, label=f"Eventos actuales ({ev})")
for _, r in pot.iterrows():
    ax.annotate(f"{r['eventos_necesarios']:.0f} ev\n(~{r['anios_aprox']:.1f} a)",
                (r["delta_auc"], r["eventos_necesarios"]),
                textcoords="offset points", xytext=(6, 4), fontsize=7.5)
ax.set_xlabel("Δ AUC mínima detectable (80 % potencia, r = 0,5)")
ax.set_ylabel("Eventos necesarios")
ax.set_title("Cuántos incumplimientos harían falta para ver una mejora")
ax.legend(loc="upper right")
figuras.guardar(fig, "fase6", "I03", "potencia_adelante",
                "Para detectar una mejora de 0,05 de AUC hacen falta cientos de eventos; con la tasa actual son varios años.")

## 2. Monitoreo retrospectivo del puntaje y de los predictores

Reconstruyo el PSI trimestral del puntaje del modelo bayesiano secuencial frente a su distribución de entrenamiento vigente, y el de `log_monto` y `log_emp_previos` (las variables que la ficha marca para vigilar).

In [4]:
bit = pd.read_csv(config.RESULTADOS / "fase6" / "bitacora_actualizaciones.csv")
bay = bit[bit["estrategia"] == "bayes_secuencial"].copy()

# Serie de PSI max y disparos
fig, ax = plt.subplots(figsize=(9.5, 3.8))
ax.plot(bay["origen"], bay["psi_max"], marker="o", ms=4, color=figuras.NARANJA, label="PSI máx. predictores")
if bay["psi_puntaje"].notna().any():
    ax.plot(bay["origen"], bay["psi_puntaje"], marker="s", ms=3.5, color=figuras.AZUL, label="PSI del puntaje")
ax.axhline(ad.PSI_UMBRAL, color=figuras.ROJO, ls="--", lw=1, label=f"Umbral {ad.PSI_UMBRAL}")
ax.set_ylabel("PSI")
ax.set_xlabel("Origen")
ax.tick_params(axis="x", rotation=45)
ax.legend(fontsize=8)
ax.set_title("Disparadores de deriva a lo largo del calendario prequential")
figuras.guardar(fig, "fase6", "I04", "psi_disparadores",
                "El PSI de predictores supera 0,25 en varios orígenes; el umbral de deriva sí se habría activado.")

## 3. Protocolo de monitoreo

Lo escribo como documento operativo en `resultados/fase6/protocolo_monitoreo.md`. Distingo con claridad el reentrenamiento por lotes (lo que haré) del aprendizaje en línea (lo que no afirmo).

In [5]:
proto = f'''# Protocolo de monitoreo y actualización del modelo (tesis v2)

**Fecha:** 15-nov-2026 · **Modelo de referencia:** M2 (elegido) / actualización bayesiana secuencial con prior de Home Credit (M3) · **Versión desplegada en AWS:** M2, v20261115.

## 1. Qué es y qué no es este protocolo

| Concepto | Qué hago | Qué no hago |
|---|---|---|
| Reentrenamiento por lotes | Reestimo coeficientes cuando se cumple un disparador, con todos los créditos cuyo desenlace ya es conocido | Actualizar el modelo con cada crédito nuevo en tiempo real |
| Aprendizaje en línea | — | No lo implemento ni lo reivindico: con ~6 eventos/año no hay señal estable crédito a crédito |
| Fecha de conocimiento | Un crédito desembolsado en t0 entra al entrenamiento en t0 + {ad.MESES_CONOCIMIENTO} meses | Usar la etiqueta antes de que madure la ventana de 12 meses |

## 2. Indicadores que sigo cada trimestre

1. **Tasa observada por banda A–E** (límites relativos a la tasa base vigente) y monotonía.
2. **PSI** de `log_monto`, `log_emp_previos`, `edad_t0` y `log_plazo` frente a la distribución del último entrenamiento.
3. **PSI del puntaje** (PD) frente a la distribución de PD del entrenamiento.
4. **Signo de los coeficientes reestimados** (ventana expansiva de diagnóstico): cambio de signo sostenido en dos trimestres.
5. **Calibración en grande:** PD media frente a tasa observada en la cohorte más reciente con desenlace conocido.

## 3. Disparadores de reentrenamiento

Se dispara un reentrenamiento por lotes si ocurre **cualquiera** de:

| Disparador | Umbral | Justificación |
|---|---|---|
| Calendario | Cada trimestre natural | Ritmo mínimo de gobierno |
| Conteo de eventos | Cada {10} incumplimientos nuevos maduros | Acumular señal antes de mover coeficientes |
| Deriva de predictores | PSI > {ad.PSI_UMBRAL} en `log_monto` o `log_emp_previos` | Variables vigiladas en la ficha del modelo |
| Deriva del puntaje | PSI del PD > {ad.PSI_UMBRAL} | Cambio de la distribución de riesgo predicida |
| Inversión de bandas | Tasa de C+D por debajo de A+B en dos trimestres | Falla de ordenamiento de negocio |

## 4. Cómo actualizo (mecanismo M3)

1. Tomo como *prior* el **posterior del modelo vigente**: media y varianza de Laplace de intercepto y coeficientes, con la varianza multiplicada por un factor de olvido de {ad.FACTOR_OLVIDO} (fijado a priori).
2. La verosimilitud usa **solo los créditos que maduraron desde la última actualización**, para no contar dos veces los datos.
3. Estimo el MAP con esos dos elementos; el primer modelo es M3 (prior de Home Credit, τ elegido en un pliegue interno).
4. Recalibro el intercepto a la tasa reciente (calibración en grande) antes de publicar.
5. Conservo M2 estático como **testigo**: si el modelo actualizado no mejora el log loss fuera de la bolsa en la ventana de control, no lo promuevo.

## 5. Buró (`veces_30mas` / `veces_90mas`)

Quedan **fuera del modelo** hasta que la operación documente su definición y existan créditos maduros con buró observado (enmienda v1.1). En monitoreo solo los perfilo: cobertura trimestral y PSI cuando la cobertura superé el 30 %.

## 6. Despliegue AWS

- Desde la versión v20261115 la API (`06_despliegue/`) califica con **M2** (11 predictores, bandas A–E relativas a la tasa base). Reemplazó al modelo v20261107 de la tesis v1, con autorización del autor y sin subir datos personales (Ley 1581).
- Cada actualización futura se publica como un nuevo `modelo.json`; el contrato se mantiene: entrada = solicitud con sus 11 predictores, salida = PD + banda, y un endpoint de retroalimentación de mora para el lote de reentrenamiento.
- Tolerancia de reproducibilidad local frente a AWS: 1e-9 en la PD.

## 7. Criterio de éxito de una actualización

Una actualización se declara exitosa solo si, en la ventana de control posterior (créditos que maduran después del corte, sin haber entrado al entrenamiento):

- el Δ log loss frente al testigo estático es ≤ 0 (punto), **y**
- el IC 95 % bootstrap por cliente del Δ no queda enteramente por encima de 0, **o** hay al menos el número de eventos que exige la tabla de potencia de la Fase 6 para el Δ buscado.

Con menos eventos que ese umbral, la actualización se registra como **exploratoria**, no como mejora demostrada.
'''
ruta = config.RESULTADOS / "fase6" / "protocolo_monitoreo.md"
ruta.write_text(proto, encoding="utf-8")
print("Escrito:", ruta)

Escrito: resultados/fase6/protocolo_monitoreo.md


## 4. Nota sobre el despliegue

Con autorización del autor (15-nov-2026), el servicio de AWS se actualizó del modelo v20261107 (tesis v1) a **M2 (v20261115)**, el modelo elegido por la regla preespecificada. Solo se publicaron parámetros, sin datos personales (Ley 1581). La actualización bayesiana secuencial queda como mecanismo para las versiones siguientes.

In [6]:
despliegue = pd.DataFrame([
    {"elemento": "API scoring", "estado": "Operativa con M2 (v20261115)", "accion_fase6": "Reemplazó a v20261107"},
    {"elemento": "Artefacto del modelo", "estado": "M2, logística L2 (11 predictores)", "accion_fase6": "Publicado con la tesis v2"},
    {"elemento": "Endpoint de mora / feedback", "estado": "Diseñado en LEEME_despliegue", "accion_fase6": "Alinearlo al protocolo de lotes"},
    {"elemento": "Contrato PD + banda", "estado": "Vigente", "accion_fase6": "Conservar si un día se exporta M2/M3"},
    {"elemento": "Tol. reproducibilidad", "estado": "1e-9", "accion_fase6": "Mantener"},
])
io_datos.guardar_resultado(despliegue, "fase6", "despliegue_referencia")
despliegue

,elemento,estado,accion_fase6
0,API scoring,Operativa con M2 (v20261115),Reemplazó a v20261107
1,Artefacto del modelo,"M2, logística L2 (11 predictores)",Publicado con la tesis v2
2,Endpoint de mora / feedback,Diseñado en LEEME_despliegue,Alinearlo al protocolo de lotes
3,Contrato PD + banda,Vigente,Conservar si un día se exporta M2/M3
4,Tol. reproducibilidad,1e-9,Mantener


## 5. Informe de la fase y Excel completo

In [7]:
informe = f'''# Informe de la Fase 6 — Componente adaptativo / incremental

Tesis v2 · Eider Johan Díaz Berrío · 15-nov-2026. Notebooks 15 y 16. Figuras I01–I04. Tablas en `p7_v2_adaptativo.xlsx`. Protocolo en `protocolo_monitoreo.md`.

## 1. Qué reformulé

Dejo de presentar el paso como "aprendizaje en línea". Lo que simulo es un **reentrenamiento por lotes** con fecha de conocimiento a 13 meses (versión 1.2, tras la auditoría): cada estrategia elige sus hiperparámetros en un pliegue interno con los datos conocidos en el corte, y la estrategia bayesiana actualiza solo con los créditos nuevos, con el posterior anterior como *prior* (factor de olvido {ad.FACTOR_OLVIDO}). Comparo cinco estrategias: la tasa base conocida, M2 congelado, M2 expansivo, M2 ponderado y la actualización bayesiana.

## 2. Resultados de la simulación prequential

| Estrategia | n | Eventos | Log loss | AUC |
|---|---:|---:|---:|---:|
''' + "\n".join(
    f"| {r.estrategia} | {int(r.n)} | {int(r.eventos)} | {r.log_loss:.4f} | {r.auc:.3f} |"
    for r in res.itertuples()
) + f'''

Comparaciones pareadas (bootstrap por cliente, 2.000 réplicas) frente al estático:

''' + "\n".join(
    f"- **{r.estrategia}**: Δ log loss = {r.delta_log_loss:+.4f} [{r.ll_ic_inf:+.4f}; {r.ll_ic_sup:+.4f}] (p = {r.ll_p:.3f}); "
    f"Δ AUC = {r.delta_auc:+.3f} [{r.auc_ic_inf:+.3f}; {r.auc_ic_sup:+.3f}] (p = {r.auc_p:.3f})."
    for r in comp[comp.frente_a == "estatico"].itertuples()
) + f'''

**Lectura.** Las cifras de la tabla y de las comparaciones pareadas son las que mandan. En la ejecución con el entorno fijado (19-nov-2026, enmienda v1.3), reentrenar (expansivo o ponderado) no mejora al M2 congelado de forma demostrable, la actualización bayesiana es peor que el congelado en log loss y en AUC, y ninguna estrategia mejora a la tasa base conocida en log loss. En una ejecución anterior con otra versión de las librerías, el congelado eligió C = 2,15, quedó sobreajustado y reentrenar parecía corregirlo: con 6 eventos, esa conclusión depende de una sola elección de hiperparámetro. Hay 10 eventos de evaluación (potencia del orden de {pot.loc[pot.delta_auc == 0.05, "potencia_actual_%"].iloc[0]:.0f} % para un Δ AUC de 0,05). La simulación es exploratoria: muestra que el mecanismo funciona sin fuga, no que actualizar mejore la calibración ni la discriminación.

## 3. Disparadores

El conteo de 10 eventos nuevos y el PSI > 0,25 sobre predictores clave se habrían activado en varios orígenes (ver `resumen_disparadores.csv` e I04). El calendario trimestral es el ritmo mínimo; el PSI aporta un freno cuando la cartera cambia de composición (sobre todo el monto y la experiencia del empleador).

## 4. Potencia hacia adelante

Tasa media observada: **{ev_anio:.1f} eventos por año** de originación.

| Δ AUC | Eventos necesarios | Adicionales | Años aprox. | Potencia actual (%) |
|---|---:|---:|---:|---:|
''' + "\n".join(
    f"| {row['delta_auc']:.2f} | {int(row['eventos_necesarios'])} | {int(row['eventos_adicionales'])} | {row['anios_aprox']:.1f} | {row['potencia_actual_%']:.1f} |"
    for _, row in pot.iterrows()
) + f'''

Para reivindicar una mejora pequeña (0,05 de AUC) hacen falta del orden de **{int(pot.loc[pot.delta_auc == 0.05, "eventos_necesarios"].iloc[0])} eventos**. Con el ritmo actual eso son varios años de operación, o un cambio de objetivo auxiliar (30+/6m) si se acepta otra definición.

## 5. Despliegue

El servicio de AWS publica M2 desde la versión v20261115, en reemplazo del modelo de la tesis v1. Las actualizaciones siguientes usan el mecanismo de este protocolo (lotes con *prior* del modelo vigente), no un peso fijo del 20 % sobre Home Credit.

## 6. Qué podemos y qué no podemos afirmar

**Podemos afirmar**

- Que la simulación respeta la fecha de conocimiento (sin fuga de desenlaces futuros).
- Que el mecanismo bayesiano secuencial está bien definido y no cuenta dos veces los datos.
- Que la lectura de la simulación es frágil: cambia con la elección del hiperparámetro del modelo congelado.
- Que el PSI y el conteo de eventos son disparadores operativos verificables.
- Que la potencia hacia adelante cuantifica cuántos eventos harían falta para detectar una mejora.

**No podemos afirmar**

- Que algún modelo, adaptado o no, mejore a la tasa base conocida en log loss.
- Que adaptar mejore la calibración o la discriminación: reentrenar no difiere del congelado de forma demostrable y la actualización bayesiana es peor.
- Que el componente sea "aprendizaje en línea".

## 7. Implicación para el documento de tesis

La adaptación se presenta como **protocolo de gobierno con evidencia de simulación exploratoria**: con los eventos actuales no demuestra mejora frente al modelo congelado ni supera a la tasa base.
'''
ruta_inf = config.RESULTADOS / "fase6" / "informe_fase6.md"
ruta_inf.write_text(informe, encoding="utf-8")
print("Informe:", ruta_inf)

Informe: resultados/fase6/informe_fase6.md


In [8]:
# Completar Excel con potencia y despliegue
wb_path = config.RESULTADOS / "fase6" / "p7_v2_adaptativo.xlsx"
with pd.ExcelWriter(wb_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as xl:
    pot.to_excel(xl, sheet_name="potencia", index=False)
    despliegue.to_excel(xl, sheet_name="despliegue", index=False)
    pd.read_csv(config.RESULTADOS / "fase6" / "resumen_disparadores.csv").to_excel(
        xl, sheet_name="disparadores_resumen", index=False)

figuras.escribir_catalogo("fase6", "Catálogo de figuras — Fase 6 (adaptativo)")

reg = io_datos.escribir_registro("16_potencia_y_monitoreo", {
    "salidas": [
        "resultados/fase6/potencia_adelante.csv",
        "resultados/fase6/protocolo_monitoreo.md",
        "resultados/fase6/informe_fase6.md",
        "resultados/fase6/despliegue_referencia.csv",
        "resultados/fase6/p7_v2_adaptativo.xlsx",
        "figuras/fase6/I03_potencia_adelante.png",
        "figuras/fase6/I04_psi_disparadores.png",
        "resultados/fase6/catalogo_figuras.md",
    ],
    "eventos_actuales": ev,
    "eventos_por_anio": ev_anio,
})
print("Registro:", reg)

Registro: evidencias/registro_16_potencia_y_monitoreo.json
